# 06 — Phase 1 Submission (Standalone Hybrid Retrieval)

This notebook is a standalone submission notebook for **Phase 1 retrieval**.

The code below is intentionally copied/adapted from a properly structured project implementation (modular source files in a full repository). The goal here is portability: the notebook can run on environments such as **Google Colab** or **Kaggle** without requiring the original project tree.


## Optional Dependency Installation

Run the next cell only if the required Python packages are not already installed in your environment.

This is especially useful on:
- Google Colab
- Kaggle notebooks
- a fresh local virtual environment

The installation cell checks for missing packages first and only installs what is needed.


In [ ]:
import importlib
import subprocess
import sys

# Map import names to pip package names.
required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "rank_bm25": "rank-bm25",
    "sentence_transformers": "sentence-transformers",
}

missing = []
for import_name, package_name in required_packages.items():
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        missing.append(package_name)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    print("Installation completed.")
else:
    print("All required packages are already installed.")


## Shared Imports

This section groups the imports that are broadly used across multiple cells in this notebook.

More specific imports (for example, library-specific retrieval components or optional runtime imports) are intentionally kept inside the relevant code cells to preserve readability and keep each section self-contained.


In [ ]:
import json
import hashlib
import sys
import re
import time
from pathlib import Path

import numpy as np


## Runtime Setup

This notebook uses a **standalone-first** path resolution strategy.

### Supported execution modes
1. **Standalone mode (recommended for Colab/Kaggle)**
   - Put the notebook and data files in the same working directory.
2. **Standalone subfolders mode**
   - Use `raw/` and `processed/` folders next to the notebook.
3. **Project mode (local repository fallback)**
   - If a project tree exists, the notebook falls back to `data/raw`, `data/processed`, and `outputs/...`.

### Expected files (standalone mode)
Preferred (faster, preprocessed):
- `docs_with_content.json`
- `queries_test_with_content.json`

Fallback (raw files, `content` is rebuilt in memory):
- `docs.json`
- `queries_test.json`


In [ ]:
cwd = Path.cwd().resolve()


def _has_files(base: Path, filenames: list[str]) -> bool:
    base = Path(base)
    return all((base / name).exists() for name in filenames)


def _resolve_runtime_paths(cwd: Path):
    """Standalone-first path resolution for local/Colab/Kaggle execution."""
    raw_needed = ["docs.json", "queries_test.json"]
    processed_needed = ["docs_with_content.json", "queries_test_with_content.json"]

    if _has_files(cwd, processed_needed) or _has_files(cwd, raw_needed):
        mode = "standalone"
        raw_dir = cwd
        processed_dir = cwd
        cache_dir = cwd / "cache"
        output_submission = cwd / "submission.csv"
        project_root = cwd
        return mode, project_root, raw_dir, processed_dir, cache_dir, output_submission

    if _has_files(cwd / "processed", processed_needed) or _has_files(cwd / "raw", raw_needed):
        mode = "standalone_subdirs"
        raw_dir = cwd / "raw"
        processed_dir = cwd / "processed"
        cache_dir = cwd / "cache"
        output_submission = cwd / "submission.csv"
        project_root = cwd
        return mode, project_root, raw_dir, processed_dir, cache_dir, output_submission

    project_root = cwd if (cwd / "src").exists() else cwd.parent
    mode = "project"
    raw_dir = project_root / "data" / "raw"
    processed_dir = project_root / "data" / "processed"
    cache_dir = project_root / "data" / "cache"
    output_submission = project_root / "outputs" / "submissions" / "submission.csv"
    return mode, project_root, raw_dir, processed_dir, cache_dir, output_submission


RUNTIME_MODE, project_root, RAW_DIR, PROCESSED_DIR, CACHE_DIR, OUTPUT_SUBMISSION = _resolve_runtime_paths(cwd)


NOTEBOOK_T0 = time.perf_counter()
print("Notebook timer started.")
print("runtime_mode:", RUNTIME_MODE)
print("cwd:", cwd)
print("project_root:", project_root)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("OUTPUT_SUBMISSION:", OUTPUT_SUBMISSION)


## Environment / Versions

This section prints the execution environment and library versions used for this run.

It improves reproducibility when the notebook is executed on different platforms (local machine, Google Colab, Kaggle).


In [ ]:
import sys
import platform
from importlib.metadata import PackageNotFoundError, version

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Executable:", sys.executable)
print("runtime_mode:", globals().get("RUNTIME_MODE", "unknown"))

packages_to_report = [
    "numpy",
    "pandas",
    "rank-bm25",
    "sentence-transformers",
    "torch",
]

for pkg in packages_to_report:
    try:
        print(f"{pkg}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg}: not installed")

try:
    import torch
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        try:
            print("CUDA device:", torch.cuda.get_device_name(0))
        except Exception:
            pass
    has_mps = bool(getattr(torch.backends, "mps", None)) and torch.backends.mps.is_available()
    print("torch.backends.mps.is_available():", has_mps)
except Exception as exc:
    print("Torch runtime check skipped:", exc)


## Configuration (Hybrid BM25 + Embeddings)

This section contains all runtime parameters used by the submission pipeline.

### What is configured here
- Retrieval output format (`TOP_K`, `CATEGORY_VALUE`)
- Data loading behavior (processed vs raw fallback)
- Embedding model and inference settings
- Hybrid fusion parameters (BM25 + embeddings with weighted RRF)

### Device recommendation
- Use `"cuda"` on Google Colab / Kaggle when a GPU runtime is enabled (NVIDIA GPU).
- Use `"cpu"` if no GPU is available.
- Use `"mps"` only on Apple Silicon Macs (Metal backend).

This cleaned notebook is intentionally focused on the **hybrid retrieval pipeline** because that is the method used in the current experiments.


In [ ]:
RETRIEVAL_METHOD = "hybrid_bm25_embeddings"  # Only method in the notebook of submission
TOP_K = 100
TEXT_FIELD = "content"
CATEGORY_VALUE = "?"

# Optional submission output override.
# None / "auto" / "" / "none" => keep environment-aware auto path.
# Example (Kaggle): "/kaggle/working/submission.csv"
OUTPUT_SUBMISSION_PATH = None

RAW_DIR_PATH = None
PROCESSED_DIR_PATH = None
CACHE_DIR_PATH = None

USE_PROCESSED_IF_AVAILABLE = True  # True | False
CLEAN_CONTENT = False               # True | False

# Keep None for full corpus.
MAX_DOCS = None                    # None | 5000 | 20000 | 50000

# Embedding model (used inside the hybrid pipeline)
EMBEDDING_MODEL_NAME = "all-MiniLM-L12-v2"   # "all-MiniLM-L12-v2" | "all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 64                     # 32 | 64 | 128 | 256
SHOW_PROGRESS_BAR = True                      # True | False

# Embedding inference tuning
EMBEDDING_DEVICE = "auto"                    # "auto" | None | "cpu" | "mps" | "cuda"
EMBEDDING_PRECISION = "float32"              # "float32" | "int8" | "uint8" | "binary" | "ubinary"
EMBEDDING_MAX_SEQ_LENGTH = None               # None | 96 | 128 | 256
EMBEDDING_TRUNCATE_DIM = None                 # None | 384 | 256
EMBEDDING_CHUNK_SIZE = None                   # None | 64 | 128 | 256
EMBEDDING_NORMALIZE = True                    # True | False
EMBEDDING_LOCAL_FILES_ONLY = False            # True for strict HF local-only mode (cache is always used first)

# Optional text truncation before retrieval (speed/quality tradeoff)
DOC_TEXT_TRUNCATE_CHARS = None                # None | 512 | 1024 | 2048
QUERY_TEXT_TRUNCATE_CHARS = None              # None | 128 | 256 | 512

# Hybrid BM25 + Embeddings settings (weighted RRF fusion)
HYBRID_BM25_METHOD = "plus"                  # "plus" | "okapi"
HYBRID_CANDIDATE_MULTIPLIER = 1               # >=1
HYBRID_RRF_K = 20                             # typical values: 10..100
HYBRID_WEIGHT_EMBEDDINGS = 3.5                # >0
HYBRID_WEIGHT_BM25 = 0.5                      # >0

def _resolve_embedding_device_config(device: str | None):
    """Return a safe device value; `None` means auto selection by the backend."""
    if device in (None, "auto"):
        return None
    try:
        import torch
    except Exception:
        print(f"[warn] torch unavailable; falling back to auto device instead of {device!r}.")
        return None
    if device == "cuda" and not torch.cuda.is_available():
        print("[warn] CUDA requested but unavailable; falling back to auto device.")
        return None
    has_mps = bool(getattr(torch.backends, "mps", None)) and torch.backends.mps.is_available()
    if device == "mps" and not has_mps:
        print("[warn] MPS requested but unavailable; falling back to auto device.")
        return None
    return device

RESOLVED_EMBEDDING_DEVICE = _resolve_embedding_device_config(EMBEDDING_DEVICE)


def _resolve_path_override_config(
    path_value: str | Path | None,
    auto_path: Path,
    cwd: Path,
    label: str,
) -> Path:
    """Resolve optional path override while preserving smart runtime defaults."""
    if path_value is None:
        return auto_path

    if isinstance(path_value, str):
        normalized = path_value.strip()
        if normalized == "" or normalized.lower() in {"auto", "none", "default"}:
            return auto_path
        path = Path(normalized)
    elif isinstance(path_value, Path):
        path = path_value
    else:
        print(
            f"[warn] Unsupported {label} type {type(path_value).__name__}; using auto path."
        )
        return auto_path

    path = path.expanduser()
    if not path.is_absolute():
        path = cwd / path
    return path


RAW_DIR_AUTO = RAW_DIR
PROCESSED_DIR_AUTO = PROCESSED_DIR
CACHE_DIR_AUTO = CACHE_DIR
OUTPUT_SUBMISSION_AUTO = OUTPUT_SUBMISSION

RAW_DIR = _resolve_path_override_config(
    RAW_DIR_PATH,
    auto_path=RAW_DIR_AUTO,
    cwd=cwd,
    label="RAW_DIR_PATH",
)
PROCESSED_DIR = _resolve_path_override_config(
    PROCESSED_DIR_PATH,
    auto_path=PROCESSED_DIR_AUTO,
    cwd=cwd,
    label="PROCESSED_DIR_PATH",
)
CACHE_DIR = _resolve_path_override_config(
    CACHE_DIR_PATH,
    auto_path=CACHE_DIR_AUTO,
    cwd=cwd,
    label="CACHE_DIR_PATH",
)
OUTPUT_SUBMISSION = _resolve_path_override_config(
    OUTPUT_SUBMISSION_PATH,
    auto_path=OUTPUT_SUBMISSION_AUTO,
    cwd=cwd,
    label="OUTPUT_SUBMISSION_PATH",
)


# Strict submission config validation (Phase 1 Kaggle format)
if TOP_K != 100:
    raise ValueError(f"Phase 1 Kaggle submission expects TOP_K=100, got {TOP_K}.")
if CATEGORY_VALUE != "?":
    raise ValueError(
        f"Phase 1 submission expects CATEGORY_VALUE='?', got {CATEGORY_VALUE!r}."
    )

print("RETRIEVAL_METHOD:", RETRIEVAL_METHOD)
print("TOP_K:", TOP_K)
print("RAW_DIR_PATH (config):", RAW_DIR_PATH)
print("RAW_DIR (auto):", RAW_DIR_AUTO)
print("RAW_DIR (final):", RAW_DIR)
print("PROCESSED_DIR_PATH (config):", PROCESSED_DIR_PATH)
print("PROCESSED_DIR (auto):", PROCESSED_DIR_AUTO)
print("PROCESSED_DIR (final):", PROCESSED_DIR)
print("CACHE_DIR_PATH (config):", CACHE_DIR_PATH)
print("CACHE_DIR (auto):", CACHE_DIR_AUTO)
print("CACHE_DIR (final):", CACHE_DIR)
print("OUTPUT_SUBMISSION_PATH (config):", OUTPUT_SUBMISSION_PATH)
print("OUTPUT_SUBMISSION (auto):", OUTPUT_SUBMISSION_AUTO)
print("OUTPUT_SUBMISSION (final):", OUTPUT_SUBMISSION)
print("USE_PROCESSED_IF_AVAILABLE:", USE_PROCESSED_IF_AVAILABLE)
print("MAX_DOCS:", MAX_DOCS)
print("EMBEDDING_MODEL_NAME:", EMBEDDING_MODEL_NAME)
print("EMBEDDING_BATCH_SIZE:", EMBEDDING_BATCH_SIZE)
print("EMBEDDING_DEVICE (requested):", EMBEDDING_DEVICE)
print("EMBEDDING_DEVICE (resolved):", RESOLVED_EMBEDDING_DEVICE or "auto")
print("EMBEDDING_PRECISION:", EMBEDDING_PRECISION)
print("EMBEDDING_MAX_SEQ_LENGTH:", EMBEDDING_MAX_SEQ_LENGTH)
print("EMBEDDING_TRUNCATE_DIM:", EMBEDDING_TRUNCATE_DIM)
print("EMBEDDING_CHUNK_SIZE:", EMBEDDING_CHUNK_SIZE)
print("EMBEDDING_NORMALIZE:", EMBEDDING_NORMALIZE)
print("EMBEDDING_LOCAL_FILES_ONLY:", EMBEDDING_LOCAL_FILES_ONLY)
print("DOC_TEXT_TRUNCATE_CHARS:", DOC_TEXT_TRUNCATE_CHARS)
print("QUERY_TEXT_TRUNCATE_CHARS:", QUERY_TEXT_TRUNCATE_CHARS)
print("HYBRID_BM25_METHOD:", HYBRID_BM25_METHOD)
print("HYBRID_CANDIDATE_MULTIPLIER:", HYBRID_CANDIDATE_MULTIPLIER)
print("HYBRID_RRF_K:", HYBRID_RRF_K)
print("HYBRID_WEIGHT_EMBEDDINGS:", HYBRID_WEIGHT_EMBEDDINGS)
print("HYBRID_WEIGHT_BM25:", HYBRID_WEIGHT_BM25)


## JSON Loading Utilities

This cell provides a minimal JSON loader used throughout the notebook.

It is intentionally small because this standalone notebook only needs the submission-time data files (`docs`, `queries_test`) and does not require the full training/evaluation loading stack.


In [ ]:
def load_json(path: Path) -> object:
    """Load a JSON file and return the parsed Python object."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


## Text Preprocessing (Copied from Project Logic)

This cell reconstructs the `content` field from raw inputs (`title`, `text`, `tags`) and optionally applies light cleaning.

It is used only when preprocessed files are not available. When `docs_with_content.json` and `queries_test_with_content.json` are present, this step is skipped.


In [ ]:
def to_string(v):
        """Convert a value to string, trim whitespace, and handle None."""
        if v is None:
            return ""
        return str(v).strip()


def build_content(item: dict) -> str:
    """
    Concatenate title + text + tags while handling missing values.

    Args:
        item : dict
            Dictionary containing 'title', 'text', and 'tags' fields.
            Example: {'id': '...', 'title': 'Some title', 'text': 'Some text', 'tags': ['tag1', 'tag2']}

    Returns:
        str
            Concatenated title + text + tags as a single string.
    """
    parts = []

    # Add title if present and non-empty.
    title = to_string(item.get("title"))
    if title:
        parts.append(title)

    # Add text if present and non-empty.
    text = to_string(item.get('text'))
    if text:
        parts.append(text)

    # Add tags if present and non-empty.
    tags = item.get("tags")

    if isinstance(tags, list):
        # Filter out empty/None tags in the list.
        valid_tags = [to_string(t) for t in tags if to_string(t)]
        if valid_tags:
            parts.append(" ".join(valid_tags))
    else:
        # Handle the case where tags is a string or another non-list value.
        tag_str = to_string(tags)
        if tag_str:
            parts.append(tag_str)

    return " ".join(parts)


def clean_text(text: str) -> str:
    """
    Light cleaning (lowercase, repeated whitespace, basic punctuation cleanup)
    to make the text easier to process for retrieval models.

    Args:
        text : str
            Text to clean (concatenated title + text + tags)

    Returns:
        str
            Cleaned text
    """
    # Convert to lowercase.
    text = text.lower()

    # Collapse repeated whitespace (including tabs/newlines) into a single space.
    text = re.sub(r'\s+', ' ', text)

    # Reduce repeated punctuation (e.g., "!!!" -> "!").
    text = re.sub(r'([!?.]){2,}', r'\1', text)

    # Trim leading and trailing whitespace.
    return text.strip()


def process_items(items: list[dict], clean: bool) -> list[dict]:
    """Internal helper to process a list of items."""
    enriched_items = []
    for item in items:
        # Copy to avoid mutating the original input.
        new_item = item.copy()

        # Build the content field.
        content = build_content(new_item)

        # Optional cleaning.
        if clean:
            content = clean_text(content)

        new_item['content'] = content
        enriched_items.append(new_item)
    return enriched_items


def add_content_field(docs: list[dict], queries: list[dict], clean: bool) -> tuple[list[dict], list[dict]]:
    """
    Add item['content'] (optionally cleaned) to docs and queries using
    build_content() and clean_text().

    Args:
        docs : list[dict]
            List of document dictionaries
        queries : list[dict]
            List of query dictionaries
        clean : bool
            If True, apply clean_text() to the generated content.

    Returns:
        tuple[list[dict], list[dict]]
            Tuple containing (enriched_docs, enriched_queries) with the
            added and optionally cleaned "content" field.
    """
    docs_enriched = process_items(docs, clean)
    queries_enriched = process_items(queries, clean)

    return docs_enriched, queries_enriched


## BM25 Retrieval Functions (Copied from Project Logic)

This cell contains the lexical retrieval component used by the hybrid pipeline:
- tokenization
- BM25 model fitting (`BM25Plus` / `BM25Okapi`)
- top-k retrieval
- mapping retrieval indices back to document IDs

In the hybrid pipeline, BM25 provides a strong lexical signal that complements semantic embeddings.


In [ ]:
from typing import List, Tuple, Dict
from rank_bm25 import BM25Okapi, BM25Plus

def tokenize(text:str) -> List[str]:
    """
    Simple tokenization: convert all text to lowercase and extract alphanumeric text fragments.
    """
    text = text.lower()
    tokens = re.findall(r"\b[a-zA-Z0-9]+\b",text)
    return tokens

def fit_bm25(docs:List[Dict], text_field: str = "content",method: str = "plus") -> object:
    """
    Builds a BM25 model (BM25Plus or Okapi) from a list of documents.
    Args:
        docs: list of dicts, each containing at least `text_field`
        text_field: key of the text field to index
        method: "plus" → BM25Plus, "okapi" → BM25Okapi

    Returns:
        BM25 object (BM25Plus or BM25Okapi)
    """
    texts = [doc[text_field] for doc in docs]
    tokenized_texts = [tokenize(text) for text in texts]

    if method == "plus":
        bm25_model = BM25Plus(tokenized_texts)
    elif method == "okapi":
        bm25_model = BM25Okapi(tokenized_texts)
    else:
        raise ValueError(f"Method must be 'plus' or 'okapi',not {method}")
    
    return bm25_model

def retrieve_bm25(bm25_model:object, docs:List[Dict], queries:List[Dict], k:int, text_field: str = "content") -> Tuple[np.ndarray, np.ndarray]:
    """
    Retrieve the top-k documents for each query.
    Args:
        bm25_model: trained BM25 model (BM25Plus or BM25Okapi)
        docs: list of dicts (same as for fit_bm25)
        queries: list of dicts, each containing at least `text_field`
        k: number of documents to return per query
        text_field: text field to use

    Returns:
        topk_indices: (n_queries, k) indices of the documents
        topk_scores: (n_queries, k) BM25 scores
    """

    nb_queries = len(queries)
    topk_indices = np.zeros((nb_queries,k),dtype=int)
    topk_scores = np.zeros((nb_queries,k),dtype = float)

    for i in range(nb_queries):
        query = queries[i]
        text_query = query[text_field]
        tokenized_query = tokenize(text_query)

        scores = np.array(bm25_model.get_scores(tokenized_query))
        topk = np.argsort(scores)[::-1][:k]

        topk_indices[i, :len(topk)] = topk
        topk_scores[i, :len(topk)] = scores[topk]

    return topk_indices, topk_scores


def map_indices_to_docids(topk_indices: np.ndarray, docs: List[dict]) -> List[List[str]]:
    """
    Convert top-k document indices into top-k document IDs.

    Args:
        topk_indices: Retrieved document indices per query.
        docs: Documents list aligned with retrieval indexing.

    Returns:
        Retrieved document IDs per query.
    """
    docs_ids = [str(d["id"]) for d in docs]
    doc_ids_list = [[docs_ids[id_line] for id_line in row] for row in topk_indices]
    
    return doc_ids_list


## Embedding Retrieval Functions (Copied from Project Logic)

This cell contains the semantic retrieval component:
- SentenceTransformer model loading
- embedding computation for documents and queries
- embedding-based top-k retrieval

It also includes safeguards for offline/cache-related model loading errors, which is useful on portable environments (Colab/Kaggle/local machines).


In [ ]:
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from sentence_transformers import SentenceTransformer


def _normalize_texts(texts: list[str] | None) -> list[str]:
    if texts is None:
        raise ValueError("texts must not be None.")
    return ["" if t is None else str(t) for t in texts]


def _texts_fingerprint(texts: list[str]) -> str:
    hasher = hashlib.sha256()
    for text in texts:
        encoded = str(text).encode("utf-8")
        hasher.update(len(encoded).to_bytes(8, byteorder="little"))
        hasher.update(encoded)
    return hasher.hexdigest()[:12]


def _load_npy_if_exists(path: Path) -> np.ndarray | None:
    if not Path(path).exists():
        return None
    return np.load(path, allow_pickle=False)


def _try_load_legacy_cache(
    cache_dir: Path,
    doc_texts: list[str],
    query_texts: list[str],
    model_name: str,
    text_field: str,
) -> tuple[np.ndarray, np.ndarray] | None:
    """Compatibility loader for older cache filenames used in this project."""
    cache_dir = Path(cache_dir)
    safe_model_name = model_name.replace("/", "_")
    safe_text_field = text_field.replace("/", "_")
    doc_fp = _texts_fingerprint(doc_texts)
    query_fp = _texts_fingerprint(query_texts)
    doc_path = cache_dir / f"docs_{safe_model_name}_{safe_text_field}_{doc_fp}.npy"
    query_path = cache_dir / f"queries_{safe_model_name}_{safe_text_field}_{query_fp}.npy"
    doc_emb = _load_npy_if_exists(doc_path)
    query_emb = _load_npy_if_exists(query_path)
    if doc_emb is None or query_emb is None:
        return None
    print("Loaded legacy embedding cache (.npy) from:")
    print("-", doc_path)
    print("-", query_path)
    return np.asarray(doc_emb), np.asarray(query_emb)


def _looks_like_offline_or_cache_error(exc: Exception) -> bool:
    """
    Heuristic detector for model-loading failures caused by offline environments
    or missing local Hugging Face cache.
    """
    message = str(exc).lower()
    exc_name = exc.__class__.__name__.lower()

    message_markers = (
        "nodename nor servname provided",
        "temporary failure in name resolution",
        "name resolution",
        "cannot send a request",
        "connection error",
        "max retries exceeded",
        "failed to establish a new connection",
        "httpsconnectionpool",
        "connecttimeout",
        "readtimeout",
        "offline",
        "could not connect",
        "not found in local cache",
        "is not a local folder",
        "repository not found",
    )
    name_markers = (
        "localentrynotfound",
        "repositorynotfound",
        "connectionerror",
        "connecttimeout",
        "readtimeout",
    )

    return any(marker in message for marker in message_markers) or any(
        marker in exc_name for marker in name_markers
    )


def _model_load_error_message(model_name: str, original_error: Exception) -> str:
    return (
        f"Failed to load SentenceTransformer model '{model_name}'. "
        "Likely cause: no network access and/or model not cached locally.\n"
        "How to fix:\n"
        "1) Preload the model once in an online environment:\n"
        "   from sentence_transformers import SentenceTransformer; "
        f"SentenceTransformer('{model_name}')\n"
        "2) Reuse the same local Hugging Face cache on this machine.\n"
        "3) Or reuse precomputed embeddings from data/cache for this dataset/model.\n"
        f"Original error: {original_error}"
    )


def _get_model(
    model_name: str,
    device: str | None = None,
    local_files_only: bool = False,
) -> "SentenceTransformer":
    # Load a fresh model instance each call (no in-memory model cache),
    # to mirror "cold-start" conditions.
    try:
        from sentence_transformers import SentenceTransformer
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "sentence-transformers is required for encode_texts/build_embeddings. "
            "Install dependencies from requirements.txt."
        ) from exc

    try:
        return SentenceTransformer(model_name, device=device, local_files_only=local_files_only)
    except Exception as exc:
        if _looks_like_offline_or_cache_error(exc):
            raise RuntimeError(_model_load_error_message(model_name, exc)) from exc
        raise


def encode_texts(
    model_name: str,
    texts: list[str],
    batch_size: int = 64,
    show_progress_bar: bool = True,
    device: str | None = None,
    precision: str = "float32",
    model_max_seq_length: int | None = None,
    truncate_dim: int | None = None,
    chunk_size: int | None = None,
    normalize_embeddings: bool = True,
    local_files_only: bool = False,
) -> np.ndarray:
    if batch_size <= 0:
        raise ValueError("batch_size must be a positive integer.")

    clean_texts = _normalize_texts(texts)
    model = _get_model(model_name, device=device, local_files_only=local_files_only)

    if model_max_seq_length is not None:
        model.max_seq_length = int(model_max_seq_length)

    emb = model.encode(
        clean_texts,
        batch_size=batch_size,
        show_progress_bar=show_progress_bar,
        precision=precision,
        convert_to_numpy=True,
        normalize_embeddings=normalize_embeddings,
        device=device,
        truncate_dim=truncate_dim,
        chunk_size=chunk_size,
    )

    emb = np.asarray(emb)
    if emb.ndim == 1:
        emb = emb.reshape(1, -1)

    return emb


def build_embeddings(
    docs: list[dict],
    queries: list[dict],
    text_field: str = "content",
    model_name: str = "all-MiniLM-L6-v2",
    batch_size: int = 64,
    show_progress_bar: bool = True,
    cache_dir: Path = Path("data/cache"),
    device: str | None = None,
    precision: str = "float32",
    model_max_seq_length: int | None = None,
    truncate_dim: int | None = None,
    chunk_size: int | None = None,
    normalize_embeddings: bool = True,
    local_files_only: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Build embeddings for docs and queries.

    1) Try legacy `.npy` cache filenames used earlier in this project.
    2) Fallback to notebook-local encoding if cache files are not available.
    """
    doc_texts = [str(d.get(text_field) or "") for d in docs]
    query_texts = [str(q.get(text_field) or "") for q in queries]

    legacy_cache = _try_load_legacy_cache(
        cache_dir=cache_dir,
        doc_texts=doc_texts,
        query_texts=query_texts,
        model_name=model_name,
        text_field=text_field,
    )
    if legacy_cache is not None:
        return legacy_cache

    del cache_dir

    doc_embeddings = encode_texts(
        model_name,
        doc_texts,
        batch_size=batch_size,
        show_progress_bar=show_progress_bar,
        device=device,
        precision=precision,
        model_max_seq_length=model_max_seq_length,
        truncate_dim=truncate_dim,
        chunk_size=chunk_size,
        normalize_embeddings=normalize_embeddings,
        local_files_only=local_files_only,
    )

    query_embeddings = encode_texts(
        model_name,
        query_texts,
        batch_size=batch_size,
        show_progress_bar=show_progress_bar,
        device=device,
        precision=precision,
        model_max_seq_length=model_max_seq_length,
        truncate_dim=truncate_dim,
        chunk_size=chunk_size,
        normalize_embeddings=normalize_embeddings,
        local_files_only=local_files_only,
    )

    return doc_embeddings, query_embeddings


def retrieve_embeddings(doc_emb: np.ndarray, query_emb: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Retrieve top-k documents per query using cosine similarity on embeddings.

    Args:
        doc_emb (np.ndarray): Document embedding matrix with shape (n_docs, dim).
        query_emb (np.ndarray): Query embedding matrix with shape (n_queries, dim).
        k (int): Number of top documents to return per query.

    Returns:
        tuple[np.ndarray, np.ndarray]:
            - topk_indices: shape (n_queries, k), document indices in docs order.
            - topk_scores: shape (n_queries, k), cosine scores aligned with topk_indices.
    """
    if k <= 0:
        raise ValueError("k must be a positive integer.")

    doc_emb = np.asarray(doc_emb, dtype=np.float32)
    query_emb = np.asarray(query_emb, dtype=np.float32)

    if doc_emb.ndim != 2 or query_emb.ndim != 2:
        raise ValueError("doc_emb and query_emb must be 2D arrays.")
    if doc_emb.shape[1] != query_emb.shape[1]:
        raise ValueError("doc_emb and query_emb must have the same embedding dimension.")
    if k > doc_emb.shape[0]:
        raise ValueError("k cannot be greater than the number of documents.")

    n_queries = query_emb.shape[0]
    if n_queries == 0:
        return (
            np.empty((0, k), dtype=np.int64),
            np.empty((0, k), dtype=np.float32),
        )

    doc_norm = np.linalg.norm(doc_emb, axis=1, keepdims=True)
    query_norm = np.linalg.norm(query_emb, axis=1, keepdims=True)
    doc_unit = doc_emb / np.clip(doc_norm, 1e-12, None)
    query_unit = query_emb / np.clip(query_norm, 1e-12, None)

    similarities = query_unit @ doc_unit.T

    topk_unsorted_idx = np.argpartition(-similarities, kth=k - 1, axis=1)[:, :k]
    topk_unsorted_scores = np.take_along_axis(similarities, topk_unsorted_idx, axis=1)
    rerank = np.argsort(-topk_unsorted_scores, axis=1)

    topk_indices = np.take_along_axis(topk_unsorted_idx, rerank, axis=1)
    topk_scores = np.take_along_axis(topk_unsorted_scores, rerank, axis=1)

    return topk_indices, topk_scores


## Submission Formatting Utilities (Copied from Project Logic)

This cell builds and writes the Kaggle submission file with the required schema:
- `query_id`
- `relevant_doc_ids` (JSON-encoded list)
- `category`

The output file is created automatically if the destination folder does not exist.


In [ ]:
from __future__ import annotations


SUBMISSION_COLUMNS = ["query_id", "relevant_doc_ids", "category"]
DEFAULT_CATEGORY = "?"
DEFAULT_TOP_K = 100


def _normalize_docids(docids: list[str], top_k: int) -> list[str]:
    if top_k <= 0:
        raise ValueError("top_k must be a positive integer.")

    seen: set[str] = set()
    normalized: list[str] = []
    for doc_id in docids:
        doc_id_str = str(doc_id)
        if doc_id_str not in seen:
            seen.add(doc_id_str)
            normalized.append(doc_id_str)

    if len(normalized) < top_k:
        raise ValueError(
            f"Each query must have at least {top_k} predicted doc IDs, got {len(normalized)}."
        )

    return normalized[:top_k]


def make_submission(
    query_ids: list[str],
    pred_docids: list[list[str]],
    top_k: int = DEFAULT_TOP_K,
    category: str = DEFAULT_CATEGORY,
):
    """
    Build a Kaggle submission DataFrame.

    Expected output columns:
    - query_id
    - relevant_doc_ids (JSON string list of doc IDs)
    - category (default '?')
    """
    if len(query_ids) != len(pred_docids):
        raise ValueError("query_ids and pred_docids must have the same length.")

    try:
        import pandas as pd
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "pandas is required to build submissions. Install dependencies first."
        ) from exc

    rows: list[dict[str, str]] = []
    for i, (qid, docs_for_query) in enumerate(zip(query_ids, pred_docids)):
        if docs_for_query is None:
            raise ValueError(f"pred_docids[{i}] is None.")

        normalized_docids = _normalize_docids(list(docs_for_query), top_k=top_k)
        rows.append(
            {
                "query_id": str(qid),
                "relevant_doc_ids": json.dumps(normalized_docids, ensure_ascii=False),
                "category": str(category),
            }
        )

    submission_df = pd.DataFrame(rows, columns=SUBMISSION_COLUMNS)
    return submission_df


def save_submission(
    query_ids: list[str],
    pred_docids: list[list[str]],
    output_path: Path = Path("outputs/submissions/submission.csv"),
    top_k: int = DEFAULT_TOP_K,
    category: str = DEFAULT_CATEGORY,
):
    """
    Create and save a Kaggle submission CSV.
    """
    submission_df = make_submission(
        query_ids=query_ids,
        pred_docids=pred_docids,
        top_k=top_k,
        category=category,
    )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    submission_df.to_csv(output_path, index=False)
    return submission_df


## Submission Pipeline Wrapper (Hybrid Only)

This cell assembles the previously defined building blocks into a single submission pipeline.

### Pipeline steps
1. Load test inputs (prefer processed files, otherwise rebuild `content` from raw files)
2. Run **hybrid retrieval** (BM25 + embeddings)
3. Fuse rankings with **weighted Reciprocal Rank Fusion (RRF)**
4. Convert predictions to the submission format
5. Save `submission.csv`

This notebook is intentionally restricted to the **hybrid BM25 + embeddings** path to keep the submission workflow clean and reproducible.


In [ ]:
def _truncate_text_field(items: list[dict], text_field: str, max_chars: int | None) -> list[dict]:
    if max_chars is None:
        return items
    if max_chars <= 0:
        raise ValueError("max_chars must be positive when provided.")

    truncated_items: list[dict] = []
    for item in items:
        updated = item.copy()
        updated[text_field] = str(updated.get(text_field) or "")[:max_chars]
        truncated_items.append(updated)
    return truncated_items


def _fuse_rankings_rrf(
    emb_indices: np.ndarray,
    bm25_indices: np.ndarray,
    k_out: int,
    rrf_k: int = 60,
    weight_embeddings: float = 1.0,
    weight_bm25: float = 1.0,
) -> tuple[np.ndarray, np.ndarray]:
    """Fuse BM25 and embedding rankings using weighted Reciprocal Rank Fusion (RRF)."""
    if k_out <= 0:
        raise ValueError("k_out must be > 0")
    if rrf_k <= 0:
        raise ValueError("rrf_k must be > 0")
    if emb_indices.shape[0] != bm25_indices.shape[0]:
        raise ValueError("emb_indices and bm25_indices must have the same number of queries")

    n_queries = emb_indices.shape[0]
    fused_indices = np.zeros((n_queries, k_out), dtype=np.int64)
    fused_scores = np.zeros((n_queries, k_out), dtype=np.float32)

    for qi in range(n_queries):
        scores: dict[int, float] = {}

        for rank, doc_idx in enumerate(emb_indices[qi], start=1):
            did = int(doc_idx)
            scores[did] = scores.get(did, 0.0) + (weight_embeddings / (rrf_k + rank))

        for rank, doc_idx in enumerate(bm25_indices[qi], start=1):
            did = int(doc_idx)
            scores[did] = scores.get(did, 0.0) + (weight_bm25 / (rrf_k + rank))

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        ranked = ranked[:k_out]

        fused_indices[qi, :len(ranked)] = [doc_id for doc_id, _ in ranked]
        fused_scores[qi, :len(ranked)] = [float(score) for _, score in ranked]

    return fused_indices, fused_scores


def load_submission_inputs(
    raw_dir: Path,
    processed_dir: Path,
    use_processed_if_available: bool = True,
    clean_content: bool = True,
    max_docs: int | None = None,
    text_field: str = "content",
    doc_text_truncate_chars: int | None = None,
    query_text_truncate_chars: int | None = None,
) -> tuple[list[dict], list[dict]]:
    """Load docs + test queries for submission, standalone-first."""
    docs_processed_path = processed_dir / "docs_with_content.json"
    queries_test_processed_path = processed_dir / "queries_test_with_content.json"

    if (
        use_processed_if_available
        and docs_processed_path.exists()
        and queries_test_processed_path.exists()
    ):
        docs = load_json(docs_processed_path)
        queries_test = load_json(queries_test_processed_path)
        print("Loaded processed docs/test queries with content.")
    else:
        docs_raw_path = raw_dir / "docs.json"
        queries_test_raw_path = raw_dir / "queries_test.json"

        if not docs_raw_path.exists() or not queries_test_raw_path.exists():
            raise FileNotFoundError(
                "Could not find submission input files. Expected either processed files "
                "(docs_with_content.json + queries_test_with_content.json) or raw files "
                "(docs.json + queries_test.json) in the configured folder(s).\n"
                f"raw_dir={raw_dir}\nprocessed_dir={processed_dir}"
            )

        docs_raw = load_json(docs_raw_path)
        queries_test_raw = load_json(queries_test_raw_path)
        docs, queries_test = add_content_field(docs_raw, queries_test_raw, clean=clean_content)
        print("Processed files not found. Rebuilt content from raw docs.json + queries_test.json.")

    if max_docs is not None:
        docs = docs[:max_docs]

    docs = _truncate_text_field(docs, text_field=text_field, max_chars=doc_text_truncate_chars)
    queries_test = _truncate_text_field(
        queries_test,
        text_field=text_field,
        max_chars=query_text_truncate_chars,
    )
    return docs, queries_test


def run_retrieval_submission(
    docs: list[dict],
    queries_test: list[dict],
    top_k: int,
    text_field: str,
    embedding_model_name: str,
    embedding_batch_size: int,
    show_progress_bar: bool,
    cache_dir: Path,
    embedding_device: str | None,
    embedding_local_files_only: bool,
    embedding_precision: str,
    embedding_max_seq_length: int | None,
    embedding_truncate_dim: int | None,
    embedding_chunk_size: int | None,
    embedding_normalize: bool,
    hybrid_bm25_method: str,
    hybrid_candidate_multiplier: int,
    hybrid_rrf_k: int,
    hybrid_weight_embeddings: float,
    hybrid_weight_bm25: float,
) -> tuple[list[list[str]], float]:
    """Hybrid retrieval: BM25 + embeddings, fused with weighted RRF."""
    if top_k <= 0:
        raise ValueError("top_k must be a positive integer.")
    if top_k > len(docs):
        raise ValueError(f"top_k={top_k} cannot be greater than number of docs={len(docs)}")
    if hybrid_candidate_multiplier <= 0:
        raise ValueError("hybrid_candidate_multiplier must be > 0")

    t0 = time.perf_counter()
    candidate_k = min(len(docs), max(top_k, top_k * int(hybrid_candidate_multiplier)))

    bm25_model = fit_bm25(docs, text_field=text_field, method=hybrid_bm25_method)
    bm25_indices, _ = retrieve_bm25(
        bm25_model,
        docs,
        queries_test,
        k=candidate_k,
        text_field=text_field,
    )

    doc_emb, query_emb = build_embeddings(
        docs,
        queries_test,
        text_field=text_field,
        model_name=embedding_model_name,
        batch_size=embedding_batch_size,
        show_progress_bar=show_progress_bar,
        cache_dir=cache_dir,
        device=embedding_device,
        local_files_only=embedding_local_files_only,
        precision=embedding_precision,
        model_max_seq_length=embedding_max_seq_length,
        truncate_dim=embedding_truncate_dim,
        chunk_size=embedding_chunk_size,
        normalize_embeddings=embedding_normalize,
    )
    emb_indices, _ = retrieve_embeddings(doc_emb, query_emb, k=candidate_k)

    topk_indices, topk_scores = _fuse_rankings_rrf(
        emb_indices=emb_indices,
        bm25_indices=bm25_indices,
        k_out=top_k,
        rrf_k=hybrid_rrf_k,
        weight_embeddings=hybrid_weight_embeddings,
        weight_bm25=hybrid_weight_bm25,
    )
    pred_docids = map_indices_to_docids(topk_indices, docs)

    elapsed = time.perf_counter() - t0
    print("retrieval_method: hybrid_bm25_embeddings")
    print("topk_indices shape:", topk_indices.shape)
    print("topk_scores shape:", topk_scores.shape)
    print(f"retrieval_elapsed_s: {elapsed:.2f}")
    return pred_docids, elapsed


def run_phase1_submission_pipeline(
    raw_dir: Path,
    processed_dir: Path,
    output_submission: Path,
    top_k: int,
    text_field: str,
    category_value: str,
    embedding_model_name: str,
    embedding_batch_size: int,
    show_progress_bar: bool,
    cache_dir: Path,
    use_processed_if_available: bool,
    clean_content: bool,
    max_docs: int | None,
    doc_text_truncate_chars: int | None,
    query_text_truncate_chars: int | None,
    embedding_device: str | None,
    embedding_local_files_only: bool,
    embedding_precision: str,
    embedding_max_seq_length: int | None,
    embedding_truncate_dim: int | None,
    embedding_chunk_size: int | None,
    embedding_normalize: bool,
    hybrid_bm25_method: str,
    hybrid_candidate_multiplier: int,
    hybrid_rrf_k: int,
    hybrid_weight_embeddings: float,
    hybrid_weight_bm25: float,
):
    docs, queries_test = load_submission_inputs(
        raw_dir=raw_dir,
        processed_dir=processed_dir,
        use_processed_if_available=use_processed_if_available,
        clean_content=clean_content,
        max_docs=max_docs,
        text_field=text_field,
        doc_text_truncate_chars=doc_text_truncate_chars,
        query_text_truncate_chars=query_text_truncate_chars,
    )

    query_ids = [str(q["id"]) for q in queries_test]

    pred_docids, elapsed = run_retrieval_submission(
        docs=docs,
        queries_test=queries_test,
        top_k=top_k,
        text_field=text_field,
        embedding_model_name=embedding_model_name,
        embedding_batch_size=embedding_batch_size,
        show_progress_bar=show_progress_bar,
        cache_dir=cache_dir,
        embedding_device=embedding_device,
        embedding_local_files_only=embedding_local_files_only,
        embedding_precision=embedding_precision,
        embedding_max_seq_length=embedding_max_seq_length,
        embedding_truncate_dim=embedding_truncate_dim,
        embedding_chunk_size=embedding_chunk_size,
        embedding_normalize=embedding_normalize,
        hybrid_bm25_method=hybrid_bm25_method,
        hybrid_candidate_multiplier=hybrid_candidate_multiplier,
        hybrid_rrf_k=hybrid_rrf_k,
        hybrid_weight_embeddings=hybrid_weight_embeddings,
        hybrid_weight_bm25=hybrid_weight_bm25,
    )

    submission_df = save_submission(
        query_ids=query_ids,
        pred_docids=pred_docids,
        output_path=output_submission,
        top_k=top_k,
        category=category_value,
    )

    print("Saved submission:", output_submission)
    print("Rows:", len(submission_df))
    return submission_df, elapsed


## Execute the Hybrid Submission Pipeline

Run this cell to generate the Phase 1 submission file using the configuration defined above.

The cell prints retrieval timing information and previews the first rows of the generated submission dataframe.


In [ ]:
if RETRIEVAL_METHOD != "hybrid_bm25_embeddings":
    raise ValueError(
        "This cleaned notebook only supports RETRIEVAL_METHOD='hybrid_bm25_embeddings'."
    )

submission_df, retrieval_elapsed = run_phase1_submission_pipeline(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    output_submission=OUTPUT_SUBMISSION,
    top_k=TOP_K,
    text_field=TEXT_FIELD,
    category_value=CATEGORY_VALUE,
    embedding_model_name=EMBEDDING_MODEL_NAME,
    embedding_batch_size=EMBEDDING_BATCH_SIZE,
    show_progress_bar=SHOW_PROGRESS_BAR,
    cache_dir=CACHE_DIR,
    use_processed_if_available=USE_PROCESSED_IF_AVAILABLE,
    clean_content=CLEAN_CONTENT,
    max_docs=MAX_DOCS,
    doc_text_truncate_chars=DOC_TEXT_TRUNCATE_CHARS,
    query_text_truncate_chars=QUERY_TEXT_TRUNCATE_CHARS,
    embedding_device=RESOLVED_EMBEDDING_DEVICE,
    embedding_local_files_only=EMBEDDING_LOCAL_FILES_ONLY,
    embedding_precision=EMBEDDING_PRECISION,
    embedding_max_seq_length=EMBEDDING_MAX_SEQ_LENGTH,
    embedding_truncate_dim=EMBEDDING_TRUNCATE_DIM,
    embedding_chunk_size=EMBEDDING_CHUNK_SIZE,
    embedding_normalize=EMBEDDING_NORMALIZE,
    hybrid_bm25_method=HYBRID_BM25_METHOD,
    hybrid_candidate_multiplier=HYBRID_CANDIDATE_MULTIPLIER,
    hybrid_rrf_k=HYBRID_RRF_K,
    hybrid_weight_embeddings=HYBRID_WEIGHT_EMBEDDINGS,
    hybrid_weight_bm25=HYBRID_WEIGHT_BM25,
)

submission_df.head(3)


## Validation and Runtime Summary

This final section verifies that the generated submission matches the expected Kaggle format and reports the total notebook runtime.

Checks include:
- file existence and non-empty output
- required submission columns
- valid JSON lists in `relevant_doc_ids`
- correct list length (`TOP_K`) for each query


In [ ]:

# Phase 1 strict checks (Kaggle format assumptions)
assert TOP_K == 100, f"Phase 1 Kaggle submission expects TOP_K=100, got {TOP_K}"
assert CATEGORY_VALUE == "?", f"Phase 1 submission expects CATEGORY_VALUE='?', got {CATEGORY_VALUE!r}"


assert OUTPUT_SUBMISSION.exists(), "submission.csv was not created"
assert OUTPUT_SUBMISSION.stat().st_size > 0, "submission.csv is empty"

assert list(submission_df.columns) == ["query_id", "relevant_doc_ids", "category"], "Submission columns are invalid"

parsed_docids = submission_df["relevant_doc_ids"].map(json.loads)

assert submission_df["category"].eq(CATEGORY_VALUE).all(), (
    "All rows in the submission must use the configured CATEGORY_VALUE."
)
assert parsed_docids.map(lambda x: isinstance(x, list)).all(), "relevant_doc_ids must be JSON lists"
assert parsed_docids.map(len).eq(TOP_K).all(), f"Each row must contain exactly TOP_K={TOP_K} doc_ids"

print("All submission checks passed.")
print("submission_path:", OUTPUT_SUBMISSION)


_t0 = globals().get("NOTEBOOK_T0")
if _t0 is not None:
    elapsed = time.perf_counter() - _t0
    minutes, seconds = divmod(int(elapsed), 60)
    print(f"total_notebook_time: {minutes}min {seconds}s")
else:
    print("NOTEBOOK_T0 not found. Execute setup cells from top to measure total time.")
